## 1. Rent data (Tenancy Services)

In [22]:
import pandas as pd

df = pd.read_csv("Data/raw/detailed-monthly-region-tenancy.csv")
print(df.columns.tolist())
print(df.shape)
df.head()

['Time Frame', 'Location Id', 'Location', 'Lodged Bonds', 'Active Bonds', 'Closed Bonds', 'Median Rent', 'Geometric Mean Rent', 'Upper Quartile Rent', 'Lower Quartile Rent', 'Log Std Dev Weekly Rent']
(7164, 11)


,Time Frame,Location Id,Location,Lodged Bonds,Active Bonds,Closed Bonds,Median Rent,Geometric Mean Rent,Upper Quartile Rent,Lower Quartile Rent,Log Std Dev Weekly Rent
0,1993-02-01,-99,ALL,"9,147",92058,"7,116",150,151,200,120,0.43
1,1993-02-01,-1,NaN,414,6507,345,130,129,165,100,0.46
2,1993-02-01,1,Northland Region,174,1731,141,120,119,150,100,0.30
3,1993-02-01,2,Auckland Region,"2,373",28833,"1,962",180,186,220,150,0.36
4,1993-02-01,3,Waikato Region,903,8253,693,120,124,155,100,0.34


In [23]:
df.info()
df['Time Frame'] = pd.to_datetime(df['Time Frame'])
print(df['Location'].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7164 entries, 0 to 7163
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Time Frame               7164 non-null   object 
 1   Location Id              7164 non-null   int64  
 2   Location                 6766 non-null   object 
 3   Lodged Bonds             7164 non-null   object 
 4   Active Bonds             7164 non-null   int64  
 5   Closed Bonds             7164 non-null   object 
 6   Median Rent              7164 non-null   int64  
 7   Geometric Mean Rent      7164 non-null   int64  
 8   Upper Quartile Rent      7164 non-null   int64  
 9   Lower Quartile Rent      7164 non-null   int64  
 10  Log Std Dev Weekly Rent  7164 non-null   float64
dtypes: float64(1), int64(6), object(4)
memory usage: 615.8+ KB
['ALL' nan 'Northland Region' 'Auckland Region' 'Waikato Region'
 'Bay of Plenty Region' 'Gisborne Region' "Hawke's Bay Region"

In [ ]:
regions = df[df['Location Id'] > 0].copy()
print(regions['Location'].unique())

['Northland Region' 'Auckland Region' 'Waikato Region'
 'Bay of Plenty Region' 'Gisborne Region' "Hawke's Bay Region"
 'Taranaki Region' 'Manawatu-Wanganui Region' 'Wellington Region'
 'West Coast Region' 'Canterbury Region' 'Otago Region' 'Southland Region'
 'Tasman Region' 'Nelson Region' 'Marlborough Region']


In [ ]:
latest = regions['Time Frame'].max()
snapshot = regions[regions['Time Frame'] == latest].copy()
print("Latest month:", latest)
print("Number of rows:", snapshot.shape[0])

Latest month: 2026-03-01 00:00:00
Number of rows: 16


In [26]:
rent = snapshot[['Location', 'Geometric Mean Rent']].copy()
rent = rent.rename(columns={'Location': 'region', 'Geometric Mean Rent': 'rent'})
rent = rent.sort_values('rent', ascending=False).reset_index(drop=True)
print(rent)

                      region  rent
0            Gisborne Region   620
1            Auckland Region   617
2       Bay of Plenty Region   613
3         Hawke's Bay Region   589
4              Tasman Region   559
5          Wellington Region   547
6            Taranaki Region   545
7           Northland Region   542
8               Otago Region   541
9             Waikato Region   527
10        Marlborough Region   519
11         Canterbury Region   518
12             Nelson Region   499
13  Manawatu-Wanganui Region   495
14          Southland Region   454
15         West Coast Region   426


## 2. Regional boundaries (Stats NZ, Regional Council 2023 generalised)

In [27]:
import geopandas as gpd

gdf = gpd.read_file("data/raw/regional-council-2023-generalised.gpkg")
gdf = gdf.to_crs(4326)          # reproject to WGS84 (longitude/latitude) for the web map
print(gdf.columns.tolist())
print(gdf.crs)

['REGC2023_V1_00', 'REGC2023_V1_00_NAME', 'REGC2023_V1_00_NAME_ASCII', 'LAND_AREA_SQ_KM', 'AREA_SQ_KM', 'Shape_Length', 'geometry']
EPSG:4326


## 3. Join rent data to boundaries

In [ ]:
boundary_names = set(gdf['REGC2023_V1_00_NAME'])
rent_names = set(rent['region'])
print("In rent data but not in boundaries:", rent_names - boundary_names)
print("In boundaries but not in rent data:", boundary_names - rent_names)

In rent data but not in boundaries: {'Manawatu-Wanganui Region'}
In boundaries but not in rent data: {'Area Outside Region', 'Manawatū-Whanganui Region'}


In [ ]:
name_fix = {"Manawatu-Wanganui Region": "Manawatū-Whanganui Region"}
rent["region"] = rent["region"].replace(name_fix)

In [30]:
merged = gdf.merge(rent, left_on="REGC2023_V1_00_NAME",
                   right_on="region", how="left")

print("Not matched:", merged[merged["rent"].isna()]["REGC2023_V1_00_NAME"].tolist())

Not matched: ['Area Outside Region']


In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)
regions_rent = merged[merged["rent"].notna()][["region", "rent", "geometry"]].copy()
regions_rent.to_file("data/processed/regions.gpkg", driver="GPKG")
print("Saved regions:", len(regions_rent))

Saved regions: 16


## 4. Employment data (Stats NZ, Business Employment Data, December 2025 quarter)

In [32]:
jobs = pd.read_csv("data/raw/business-employment-data-december-2025-quarter/machine-readable-business-employment-data-dec-2025-quarter.csv")
print(jobs.columns.tolist())
print(jobs.shape)
print("Groups (sub-tables):")
print(jobs['Group'].unique())
print("\nSeries_title_1 (variables):")
print(jobs['Series_title_1'].unique())

['Series_reference', 'Period', 'Data_value', 'Suppressed', 'STATUS', 'UNITS', 'Magnitude', 'Subject', 'Group', 'Series_title_1', 'Series_title_2', 'Series_title_3', 'Series_title_4', 'Series_title_5']
(25672, 14)
Groups (sub-tables):
['Industry by employment variable' 'Sex by employment variable'
 'Age by employment variable' 'Region by employment variable'
 'Territorial authority by employment variable']

Series_title_1 (variables):
['Filled jobs' 'Total earnings' 'Filled jobs (workplace location based)']


In [33]:
region = jobs[jobs['Group'] == 'Region by employment variable'].copy()

print("Variable (Series_title_1):", region['Series_title_1'].unique())
print("Region (Series_title_2):", region['Series_title_2'].unique())
print("Series_title_3:", region['Series_title_3'].unique())
print("Period range:", region['Period'].min(), "to", region['Period'].max())

Variable (Series_title_1): ['Filled jobs' 'Total earnings' 'Filled jobs (workplace location based)']
Region (Series_title_2): ['Northland' 'Auckland' 'Waikato' 'Bay of Plenty' 'Gisborne' "Hawke's Bay"
 'Taranaki' 'Manawatu-Whanganui' 'Wellington' 'Tasman' 'Nelson'
 'Marlborough' 'West Coast' 'Canterbury' 'Otago' 'Southland']
Series_title_3: ['Actual' 'Seasonally adjusted' 'Trend']
Period range: 2011.06 to 2025.12


In [ ]:
latest_period = region['Period'].max()
print("Latest quarter:", latest_period)

jobs_latest = region[
    (region['Series_title_1'] == 'Filled jobs (workplace location based)') &
    (region['Series_title_3'] == 'Actual') &
    (region['Period'] == latest_period)
].copy()

jobs_clean = jobs_latest[['Series_title_2', 'Data_value']].rename(
    columns={'Series_title_2': 'jobs_region', 'Data_value': 'jobs'}
).sort_values('jobs', ascending=False).reset_index(drop=True)

print("Rows:", len(jobs_clean), "(should be 16)")
print(jobs_clean)

Latest quarter: 2025.12
Rows: 16 (should be 16)
           jobs_region      jobs
0             Auckland  807409.0
1           Canterbury  312364.0
2           Wellington  268511.0
3              Waikato  202651.0
4        Bay of Plenty  135600.0
5                Otago  113737.0
6   Manawatu-Whanganui   98207.0
7          Hawke's Bay   74203.0
8            Northland   61420.0
9             Taranaki   49014.0
10           Southland   46781.0
11              Nelson   25457.0
12         Marlborough   21654.0
13              Tasman   20763.0
14            Gisborne   20373.0
15          West Coast   13492.0


## 5. Join employment data

In [ ]:
import unicodedata

regions = gpd.read_file("data/processed/regions.gpkg")

def norm_region(name):
    s = unicodedata.normalize('NFKD', str(name))   
    s = ''.join(c for c in s if not unicodedata.combining(c))  
    return s.lower().replace(' region', '').strip()           

regions['key'] = regions['region'].apply(norm_region)
jobs_clean['key'] = jobs_clean['jobs_region'].apply(norm_region)

print("In regions but not in jobs:", set(regions['key']) - set(jobs_clean['key']))
print("In jobs but not in regions:", set(jobs_clean['key']) - set(regions['key']))

regions = regions.merge(jobs_clean[['key', 'jobs']], on='key', how='left')
print(regions[['region', 'rent', 'jobs']])

In regions but not in jobs: set()
In jobs but not in regions: set()
                       region   rent      jobs
0            Northland Region  542.0   61420.0
1             Auckland Region  617.0  807409.0
2              Waikato Region  527.0  202651.0
3        Bay of Plenty Region  613.0  135600.0
4             Gisborne Region  620.0   20373.0
5          Hawke's Bay Region  589.0   74203.0
6             Taranaki Region  545.0   49014.0
7   Manawatū-Whanganui Region  495.0   98207.0
8           Wellington Region  547.0  268511.0
9           West Coast Region  426.0   13492.0
10          Canterbury Region  518.0  312364.0
11               Otago Region  541.0  113737.0
12           Southland Region  454.0   46781.0
13              Tasman Region  559.0   20763.0
14              Nelson Region  499.0   25457.0
15         Marlborough Region  519.0   21654.0


## 6. Normalisation and suitability score

In [ ]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

regions["jobs_score"] = normalize(regions["jobs"])

regions["rent_score"] = 1 - normalize(regions["rent"])

print(regions[["region", "rent", "rent_score", "jobs", "jobs_score"]]
      .sort_values("jobs_score", ascending=False))

                       region   rent  rent_score      jobs  jobs_score
1             Auckland Region  617.0    0.015464  807409.0    1.000000
10          Canterbury Region  518.0    0.525773  312364.0    0.376452
8           Wellington Region  547.0    0.376289  268511.0    0.321216
2              Waikato Region  527.0    0.479381  202651.0    0.238260
3        Bay of Plenty Region  613.0    0.036082  135600.0    0.153804
11               Otago Region  541.0    0.407216  113737.0    0.126266
7   Manawatū-Whanganui Region  495.0    0.644330   98207.0    0.106705
5          Hawke's Bay Region  589.0    0.159794   74203.0    0.076470
0            Northland Region  542.0    0.402062   61420.0    0.060369
6             Taranaki Region  545.0    0.386598   49014.0    0.044743
12           Southland Region  454.0    0.855670   46781.0    0.041930
14              Nelson Region  499.0    0.623711   25457.0    0.015071
15         Marlborough Region  519.0    0.520619   21654.0    0.010281
13    

In [ ]:
w_rent = 0.5
w_jobs = 0.5
regions["suitability"] = w_rent * regions["rent_score"] + w_jobs * regions["jobs_score"]

print(regions[["region", "rent_score", "jobs_score", "suitability"]]
      .sort_values("suitability", ascending=False).reset_index(drop=True))

                       region  rent_score  jobs_score  suitability
0             Auckland Region    0.015464    1.000000     0.507732
1           West Coast Region    1.000000    0.000000     0.500000
2           Canterbury Region    0.525773    0.376452     0.451113
3            Southland Region    0.855670    0.041930     0.448800
4   Manawatū-Whanganui Region    0.644330    0.106705     0.375518
5              Waikato Region    0.479381    0.238260     0.358821
6           Wellington Region    0.376289    0.321216     0.348752
7               Nelson Region    0.623711    0.015071     0.319391
8                Otago Region    0.407216    0.126266     0.266741
9          Marlborough Region    0.520619    0.010281     0.265450
10           Northland Region    0.402062    0.060369     0.231215
11            Taranaki Region    0.386598    0.044743     0.215670
12              Tasman Region    0.314433    0.009158     0.161796
13         Hawke's Bay Region    0.159794    0.076470     0.11

In [ ]:
import numpy as np

regions["jobs_score"] = normalize(np.log(regions["jobs"]))

regions["suitability"] = 0.5 * regions["rent_score"] + 0.5 * regions["jobs_score"]

result = regions[["region", "rent_score", "jobs", "jobs_score", "suitability"]] \
    .sort_values("suitability", ascending=False).reset_index(drop=True)
print(result)

                       region  rent_score      jobs  jobs_score  suitability
0           Canterbury Region    0.525773  312364.0    0.767907     0.646840
1            Southland Region    0.855670   46781.0    0.303876     0.579773
2              Waikato Region    0.479381  202651.0    0.662162     0.570771
3   Manawatū-Whanganui Region    0.644330   98207.0    0.485120     0.564725
4           Wellington Region    0.376289  268511.0    0.730936     0.553612
5             Auckland Region    0.015464  807409.0    1.000000     0.507732
6           West Coast Region    1.000000   13492.0    0.000000     0.500000
7                Otago Region    0.407216  113737.0    0.521000     0.464108
8               Nelson Region    0.623711   25457.0    0.155165     0.389438
9            Northland Region    0.402062   61420.0    0.370415     0.386238
10            Taranaki Region    0.386598   49014.0    0.315272     0.350935
11         Marlborough Region    0.520619   21654.0    0.115622     0.318120

## 7. Sensitivity analysis (70/30, 50/50, 30/70)

In [39]:
ranks = {}
for w in [0.7, 0.5, 0.3]:
    s = w * regions["rent_score"] + (1 - w) * regions["jobs_score"]
    ranks[f"{int(w*100)}/{int((1-w)*100)}"] = s.rank(ascending=False).astype(int)
table = pd.DataFrame(ranks)
table.index = regions["region"].str.replace(" Region", "")
print(table.sort_values("50/50"))

                    70/30  50/50  30/70
region                                 
Canterbury              3      1      2
Southland               2      2      7
Waikato                 5      3      4
Manawatū-Whanganui      4      4      5
Wellington              7      5      3
Auckland               12      6      1
West Coast              1      7     12
Otago                   8      8      6
Nelson                  6      9     13
Northland              10     10      9
Taranaki               11     11     11
Marlborough             9     12     14
Bay of Plenty          15     13      8
Hawke's Bay            14     14     10
Tasman                 13     15     15
Gisborne               16     16     16


## 8. Export GeoJSON for the web map

Only the fields needed by the web page are exported. The suitability score is not stored, because the browser calculates it from the user's weight.

In [1]:
export = regions[["region", "rent", "jobs", "rent_score", "jobs_score", "geometry"]].copy()
export = export.to_crs(4326)
export.to_file("data/processed/regions.geojson", driver="GeoJSON")

print("Exported fields:", export.columns.tolist())
print("Number of regions:", len(export))

NameError: name 'regions' is not defined

In [ ]:
import json
with open("data/processed/regions.geojson") as f:
    gj = json.load(f)

print("Type:", gj["type"])
print("Number of regions:", len(gj["features"]))
print("Properties of the first region:")
print(gj["features"][0]["properties"])

Type: FeatureCollection
Number of regions: 16
Properties of the first region:
{'region': 'Northland Region', 'rent': 542.0, 'jobs': 61420.0, 'rent_score': 0.4020618556701031, 'jobs_score': 0.37041479381807235}
